# 01 · Tokenizer and data (CPU notebook, no GPU quota)

Streams EN (FineWeb-Edu `sample-10BT`) and ES/FR (FineWeb2-HQ `spa_Latn`/`fra_Latn`) from the Hugging Face Hub, trains the shared 32k byte-level BPE, measures fertility against GPT-2 and writes uint16 token shards.

**Settings:** Accelerator = None, Internet = On. Run with *Save Version → Save & Run All*. The output (`/kaggle/working/data`) is the input of notebooks 02–04.

If a session is too short for all three languages, set `LANGS` to a subset and run again: finished languages are skipped (`meta_<lang>.json`), and the outputs of several versions can be attached together.

In [ ]:
# --- Get the Trilingua code -------------------------------------------------------
# Option A (after the repo is public): clone it.
# Option B: upload the repo folder as a Kaggle Dataset named "trilingua-code" and attach it.
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/aladewealth/trilingua.git"   # TODO: confirm once the repo is pushed
CODE = "/tmp/trilingua"
attached = glob.glob("/kaggle/input/**/trilingua/model.py", recursive=True)
if os.path.exists(CODE):
    pass
elif attached:
    shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), CODE)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE); sys.path.insert(0, CODE)
print("code at", CODE, os.listdir(CODE))

In [ ]:
!pip -q install -U 'tokenizers>=0.19' 'datasets>=3.0' 'huggingface_hub>=0.24' 2>&1 | tail -1
!nproc; free -g; df -h /kaggle/working | tail -1

In [ ]:
OUT = '/kaggle/working/data'
LANGS = ['en', 'es', 'fr']
TOTAL_TOKENS = 1.3e9        # ~1.25B used by pretraining + a small surplus
MB_PER_LANG = 150           # balanced tokenizer sample
os.makedirs(OUT, exist_ok=True)

## 1. Balanced raw-text sample for the tokenizer

In [ ]:
!python -m trilingua.prepare_data sample --out_dir {OUT} --langs {' '.join(LANGS)} --mb_per_lang {MB_PER_LANG}

## 2. Train the 32k byte-level BPE (special tokens: `<|endoftext|> <|pad|> <|en|> <|es|> <|fr|> <|system|> <|user|> <|assistant|> <|end|>`)

In [ ]:
if not os.path.exists(f'{OUT}/tokenizer.json'):
    !python -m trilingua.prepare_data tokenizer --out_dir {OUT} --vocab_size 32768
from trilingua.tokenizer import TrilinguaTokenizer
tok = TrilinguaTokenizer.from_file(f'{OUT}/tokenizer.json')
for s in ['¿Qué tal, señor Muñoz?', '« Ça va ? » dit le garçon à l’hôtel.', "It's a test."]:
    ids = tok.encode(s); assert tok.decode(ids) == s
    print(len(ids), [tok.tok.id_to_token(i) for i in ids])

## 3. Tokenize to shards (three languages in parallel; ~50/25/25 mix)

In [ ]:
import subprocess, time
t0 = time.time()
procs = [subprocess.Popen(['python', '-m', 'trilingua.prepare_data', 'shards', '--out_dir', OUT, '--langs', l,
                           '--total_tokens', str(TOTAL_TOKENS), '--val_tokens', '2e6'],
                          stdout=open(f'/kaggle/working/shards_{l}.log', 'w'), stderr=subprocess.STDOUT)
         for l in LANGS]
while any(p.poll() is None for p in procs):
    time.sleep(300)
    for l in LANGS:
        lines = open(f'/kaggle/working/shards_{l}.log').read().strip().splitlines()
        print(f'[{(time.time()-t0)/60:.0f} min]', lines[-1] if lines else '')
print([p.returncode for p in procs])
!python -m trilingua.prepare_data shards --out_dir {OUT}   # writes the combined meta.json

## 4. Fertility vs GPT-2 (tokens per word on held-out docs)

In [ ]:
!python -m trilingua.prepare_data fertility --out_dir {OUT} --json_out {OUT}/fertility.json --png_out {OUT}/fertility.png
from IPython.display import Image; Image(f'{OUT}/fertility.png')

In [ ]:
# keep the output small: the raw tokenizer sample is no longer needed
for f in glob.glob(f'{OUT}/tok_sample_*'):
    os.remove(f)
!du -sh {OUT}; ls -la {OUT}